In [1]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = Path("../data/raw")
jobs = pd.read_csv(DATA_DIR / "naukri_com-job_sample.csv")

print("Available columns:", jobs.columns.tolist())

Available columns: ['company', 'education', 'experience', 'industry', 'jobdescription', 'jobid', 'joblocation_address', 'jobtitle', 'numberofpositions', 'payrate', 'postdate', 'site_name', 'skills', 'uniq_id']


In [2]:
def find_description_column(columns):
    for col in columns:
        name = str(col).lower().replace("_", " ").strip()
        if "description" in name:
            return col
    return None

description_col = find_description_column(jobs.columns)

if description_col is None:
    raise ValueError("Job description column not found.")

jobs = jobs.dropna(subset=[description_col]).copy()
jobs[description_col] = jobs[description_col].astype(str)

print("Using column:", description_col)
display(jobs[[description_col]].head())

Using column: jobdescription


,jobdescription
0,Job Description Send me Jobs like this Quali...
1,Job Description Send me Jobs like this Quali...
2,Job Description Send me Jobs like this - as ...
3,Job Description Send me Jobs like this - Inv...
4,Job Description Send me Jobs like this Pleas...


In [3]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

job_matrix = vectorizer.fit_transform(jobs[description_col])

def predict_fit(resume_text, top_n=5):
    resume_vector = vectorizer.transform([resume_text])

    scores = cosine_similarity(
        resume_vector, job_matrix
    ).flatten()

    results = jobs.copy()
    results["Fit Score (%)"] = (scores * 100).round(2)

    return results.sort_values(
        "Fit Score (%)", ascending=False
    ).head(top_n)

In [4]:
sample_resume = """
Python, SQL, pandas, data analysis, machine learning,
statistics and problem-solving experience.
"""

results = predict_fit(sample_resume)

display(results)

,company,education,experience,industry,jobdescription,jobid,joblocation_address,jobtitle,numberofpositions,payrate,postdate,site_name,skills,uniq_id,Fit Score (%)
10864,Career Maker,UG: Any Graduate - Any Specialization PG:Any P...,2 - 7 yrs,IT-Software / Software Services,Job Description Send me Jobs like this Major...,121215900685,Bengaluru/Bangalore,Sr Scientist/ NLP/ Machine Learning/ Data Mini...,NaN,Not Disclosed by Recruiter,2016-03-28 17:04:48 +0000,NaN,Analytics & Business Intelligence,15480ee936aa01b23e3d3a2979d07cb3,49.80
15561,Career Maker,UG: Any Graduate - Any Specialization PG:M.Tec...,4 - 9 yrs,IT-Software / Software Services,Job Description Send me Jobs like this Major...,180716900576,Bengaluru/Bangalore,Machine Learning Scientist,NaN,Not Disclosed by Recruiter,2016-08-05 18:43:53 +0000,NaN,IT Software - Embedded,1e1c4d5d2150d7cf7084dba8ad8a7088,43.69
15615,Premium hiring for Samaksh Consulting,UG: Any Graduate - Any Specialization PG:Any P...,4 - 9 yrs,IT-Software / Software Services,Job Description Send me Jobs like this - We ...,60816000863,"Bengaluru/Bangalore , Hyderabad / Secunderabad",Technical Lead - Python Developer - SQL,NaN,Not Disclosed by Recruiter,2016-08-06 18:43:59 +0000,NaN,IT Software - DBA,798ae550974744f4e13cab276be292a0,42.29
21993,Pin Click - Startup,UG: B.Tech/B.E. - Any Specialization Doctorate...,3 - 4 yrs,Internet / Ecommerce,Job Description Send me Jobs like this * I a...,251116900622,Bengaluru,Developer Engineer and Data Scientist,NaN,Not Disclosed by Recruiter,2016-11-25 22:47:05 +0000,www.naukri.com,Analytics & Business Intelligence,d0c8d299e4d513ad4c563bc6ecfdaa00,41.76
17442,QUORA,UG: B.Tech/B.E. PG:M.Tech Doctorate:Any Doctor...,2 - 5 yrs,IT-Software / Software Services,Job Description Send me Jobs like this espon...,81015500170,Mumbai,Software Engineer - Machine Learning,NaN,Not Disclosed by Recruiter,2015-12-29 02:24:09 +0000,NaN,IT Software - Telecom Software,a2392a4cca9a0839e2b9dc137e0e58f3,40.35


In [5]:
best_score = results["Fit Score (%)"].max()

print(f"Highest text similarity score: {best_score:.2f}%")
print("This is a text similarity estimate, not a hiring probability.")

Highest text similarity score: 49.80%
This is a text similarity estimate, not a hiring probability.
